# Web Analytics URL Measurement Audit Pipeline

A public-safe, fully reproducible notebook for auditing **URL-level measurement quality** and prioritizing remediation work by traffic impact.

This project demonstrates a measurement-reliability workflow:

1. canonicalize raw URLs while removing tracking noise,
2. preserve traffic volume while aggregating duplicate URL variants,
3. join browser/network inspection results,
4. compare observed measurement fields against explicit expectations,
5. separate true tagging defects from broken URLs and uncertain captures,
6. group related defects into actionable remediation units,
7. choose a representative high-traffic URL for each issue group,
8. validate row/traffic conservation with explicit assertions.

> **Public-data note:** all domains, URLs, traffic counts, field names, and inspection outcomes in this notebook are synthetic or generalized. No production identifiers, customer data, internal URLs, credentials, proprietary scripts, or real output values are included.

## Why this matters

A metric anomaly is not automatically a business anomaly. Before using behavioral data for analysis, experimentation, or modeling, the underlying measurement must be reliable enough to support the decision.

This notebook focuses on the question:

> **Which observed data-quality failures are real, reproducible measurement defects, and which URLs should be fixed first?**


## 1. Setup and synthetic source data

The source extract intentionally contains:

- duplicate logical pages with tracking parameters,
- multiple markets and page types,
- malformed / dead URLs,
- traffic weights (`instances`),
- historical values that may or may not match the current browser/network observation.

The notebook is standalone and does not make external network calls.


In [1]:
from urllib.parse import parse_qsl, unquote, urlencode, urlsplit, urlunsplit
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)

raw_df = pd.DataFrame(
    [
        # Same logical page with tracking-noise variants
        ["https://www.example-shop.com/us/support/phones/?utm_source=email", 4200, "support", "support"],
        ["https://www.example-shop.com/us/support/phones/?cid=spring", 1800, "support", "support"],
        ["https://www.example-shop.com/us/support/phones/", 2400, "support", "support"],

        ["https://www.example-shop.com/de/shop/smartphones/?utm_campaign=launch", 7600, "product", "shop"],
        ["https://www.example-shop.com/de/shop/smartphones/?gclid=abc", 2100, "product", "shop"],

        ["https://www.example-shop.com/sg/offers/student/", 5100, "offers", "offers"],
        ["https://www.example-shop.com/sg/offers/student/?ref=partner", 900, "offers", "offers"],

        ["https://www.example-shop.com/au/apps/wallet/", 3300, "apps", "apps"],
        ["https://www.example-shop.com/au/info/privacy/", 1600, "info", "info"],

        # Dead / uncertain examples
        ["https://www.example-shop.com/us/support/legacy-device/", 700, "support", "support"],
        ["https://www.example-shop.com/de/unknown/landing/", 450, None, None],
        ["not a valid url", 120, None, None],
    ],
    columns=[
        "original_url",
        "instances",
        "historical_page_track",
        "historical_site_section",
    ],
)

display(raw_df)


,original_url,instances,historical_page_track,historical_site_section
0,https://www.example-shop.com/us/support/phones/?utm_source=email,4200,support,support
1,https://www.example-shop.com/us/support/phones/?cid=spring,1800,support,support
2,https://www.example-shop.com/us/support/phones/,2400,support,support
3,https://www.example-shop.com/de/shop/smartphones/?utm_campaign=launch,7600,product,shop
4,https://www.example-shop.com/de/shop/smartphones/?gclid=abc,2100,product,shop
5,https://www.example-shop.com/sg/offers/student/,5100,offers,offers
6,https://www.example-shop.com/sg/offers/student/?ref=partner,900,offers,offers
7,https://www.example-shop.com/au/apps/wallet/,3300,apps,apps
8,https://www.example-shop.com/au/info/privacy/,1600,info,info
9,https://www.example-shop.com/us/support/legacy-device/,700,support,support


## 2. URL canonicalization and path-feature extraction

The canonicalization rule removes **known tracking parameters** while preserving non-tracking query parameters that may change page behavior.

Path semantics used in this generalized example:

- first segment → `market`
- second segment → `page_section`
- third segment → `page_family`
- expected site-section value → second path segment

The rule is explicit and testable rather than inferred after seeing the result.


In [2]:
TRACKING_KEYS = {
    "cid", "gclid", "fbclid", "dclid", "msclkid",
    "affiliate", "affiliatename", "siteid", "source", "campaign",
}


def is_tracking_key(key: str) -> bool:
    key = str(key).strip().lower()
    return key.startswith("utm_") or key in TRACKING_KEYS


def normalize_url(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if not text:
        return pd.NA

    # Keep malformed inputs visible for downstream triage.
    if "://" not in text and not text.startswith("/"):
        return text

    if text.startswith("/"):
        text = "https://www.example-shop.com" + text

    try:
        parts = urlsplit(text)
    except Exception:
        return text

    if not parts.hostname:
        return text

    scheme = (parts.scheme or "https").lower()
    host = parts.hostname.lower()
    path = re.sub(r"/{2,}", "/", parts.path or "/")

    remaining_query = [
        (key, value)
        for key, value in parse_qsl(parts.query, keep_blank_values=True)
        if not is_tracking_key(key)
    ]
    remaining_query.sort(key=lambda item: (item[0].lower(), item[1]))

    return urlunsplit(
        (scheme, host, path, urlencode(remaining_query, doseq=True), "")
    )


def extract_path_features(value):
    result = {
        "detail_path": pd.NA,
        "market": pd.NA,
        "page_section": pd.NA,
        "page_family": pd.NA,
        "expected_site_section": pd.NA,
    }

    if pd.isna(value):
        return pd.Series(result)

    try:
        parts = urlsplit(str(value))
        if not parts.hostname:
            return pd.Series(result)

        path = re.sub(r"/{2,}", "/", parts.path or "/")
        segments = [
            unquote(segment).strip()
            for segment in path.strip("/").split("/")
            if segment
        ]
    except Exception:
        return pd.Series(result)

    result["detail_path"] = path

    if len(segments) >= 1:
        result["market"] = segments[0]
    if len(segments) >= 2:
        result["page_section"] = segments[1]
        result["expected_site_section"] = segments[1]
    if len(segments) >= 3:
        result["page_family"] = segments[2]

    return pd.Series(result)


working_df = raw_df.copy()
working_df["instances"] = pd.to_numeric(working_df["instances"], errors="coerce").fillna(0)
working_df["normalized_url"] = working_df["original_url"].apply(normalize_url)

path_features = working_df["normalized_url"].apply(extract_path_features)
working_df = pd.concat([working_df, path_features], axis=1)

display(
    working_df[
        [
            "original_url",
            "normalized_url",
            "market",
            "page_section",
            "page_family",
            "expected_site_section",
            "instances",
        ]
    ]
)


,original_url,normalized_url,market,page_section,page_family,expected_site_section,instances
0,https://www.example-shop.com/us/support/phones/?utm_source=email,https://www.example-shop.com/us/support/phones/,us,support,phones,support,4200
1,https://www.example-shop.com/us/support/phones/?cid=spring,https://www.example-shop.com/us/support/phones/,us,support,phones,support,1800
2,https://www.example-shop.com/us/support/phones/,https://www.example-shop.com/us/support/phones/,us,support,phones,support,2400
3,https://www.example-shop.com/de/shop/smartphones/?utm_campaign=launch,https://www.example-shop.com/de/shop/smartphones/,de,shop,smartphones,shop,7600
4,https://www.example-shop.com/de/shop/smartphones/?gclid=abc,https://www.example-shop.com/de/shop/smartphones/,de,shop,smartphones,shop,2100
5,https://www.example-shop.com/sg/offers/student/,https://www.example-shop.com/sg/offers/student/,sg,offers,student,offers,5100
6,https://www.example-shop.com/sg/offers/student/?ref=partner,https://www.example-shop.com/sg/offers/student/?ref=partner,sg,offers,student,offers,900
7,https://www.example-shop.com/au/apps/wallet/,https://www.example-shop.com/au/apps/wallet/,au,apps,wallet,apps,3300
8,https://www.example-shop.com/au/info/privacy/,https://www.example-shop.com/au/info/privacy/,au,info,privacy,info,1600
9,https://www.example-shop.com/us/support/legacy-device/,https://www.example-shop.com/us/support/legacy-device/,us,support,legacy-device,support,700


## 3. Aggregate only truly equivalent normalized URLs

Two source rows are collapsed only when their canonical URL and extracted path context are the same.

Traffic weight and source coverage are preserved so that later remediation can be prioritized by business impact.


In [3]:
def join_unique_limited(series, limit=10):
    values = []
    for value in series:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in values:
            values.append(text)
        if len(values) >= limit:
            values.append("...[truncated]")
            break
    return " | ".join(values) if values else pd.NA


audit_input_df = (
    working_df
    .groupby(
        [
            "normalized_url",
            "detail_path",
            "market",
            "page_section",
            "page_family",
            "expected_site_section",
        ],
        dropna=False,
        as_index=False,
    )
    .agg(
        instances=("instances", "sum"),
        source_row_count=("original_url", "size"),
        original_url_count=("original_url", "nunique"),
        original_url_examples=("original_url", join_unique_limited),
        historical_page_track=("historical_page_track", join_unique_limited),
        historical_site_section=("historical_site_section", join_unique_limited),
    )
    .sort_values("instances", ascending=False)
    .reset_index(drop=True)
)

assert audit_input_df["normalized_url"].is_unique
assert np.isclose(working_df["instances"].sum(), audit_input_df["instances"].sum())

display(audit_input_df)


,normalized_url,detail_path,market,page_section,page_family,expected_site_section,instances,source_row_count,original_url_count,original_url_examples,historical_page_track,historical_site_section
0,https://www.example-shop.com/de/shop/smartphones/,/de/shop/smartphones/,de,shop,smartphones,shop,9700,2,2,https://www.example-shop.com/de/shop/smartphones/?utm_campaign=launch | https://www.example-shop.com/de/shop/smartph...,product,shop
1,https://www.example-shop.com/us/support/phones/,/us/support/phones/,us,support,phones,support,8400,3,3,https://www.example-shop.com/us/support/phones/?utm_source=email | https://www.example-shop.com/us/support/phones/?c...,support,support
2,https://www.example-shop.com/sg/offers/student/,/sg/offers/student/,sg,offers,student,offers,5100,1,1,https://www.example-shop.com/sg/offers/student/,offers,offers
3,https://www.example-shop.com/au/apps/wallet/,/au/apps/wallet/,au,apps,wallet,apps,3300,1,1,https://www.example-shop.com/au/apps/wallet/,apps,apps
4,https://www.example-shop.com/au/info/privacy/,/au/info/privacy/,au,info,privacy,info,1600,1,1,https://www.example-shop.com/au/info/privacy/,info,info
5,https://www.example-shop.com/sg/offers/student/?ref=partner,/sg/offers/student/,sg,offers,student,offers,900,1,1,https://www.example-shop.com/sg/offers/student/?ref=partner,offers,offers
6,https://www.example-shop.com/us/support/legacy-device/,/us/support/legacy-device/,us,support,legacy-device,support,700,1,1,https://www.example-shop.com/us/support/legacy-device/,support,support
7,https://www.example-shop.com/de/unknown/landing/,/de/unknown/landing/,de,unknown,landing,unknown,450,1,1,https://www.example-shop.com/de/unknown/landing/,<NA>,<NA>
8,not a valid url,NaN,NaN,NaN,NaN,NaN,120,1,1,not a valid url,<NA>,<NA>


## 4. Browser/network inspection results

In a production workflow, this table could come from Playwright/Selenium/network instrumentation.

For a public portfolio repository, the capture layer is deliberately abstracted. The synthetic inspection table below represents the **first page-view observation** for each normalized URL.

This keeps the analytical decision logic reproducible without exposing proprietary request signatures, internal capture rules, or production endpoints.


In [4]:
inspection_df = pd.DataFrame(
    [
        # normalized_url, status, first PV?, observed page-track, observed site-section, note
        ["https://www.example-shop.com/us/support/phones/", "Live", True, "support", "support", "expected capture"],
        ["https://www.example-shop.com/de/shop/smartphones/", "Live", True, "product", "product", "site-section mismatch"],
        ["https://www.example-shop.com/sg/offers/student/", "Live", True, None, "offers", "page-track missing"],
        ["https://www.example-shop.com/sg/offers/student/?ref=partner", "Live", True, "offers", "offers", "query changes page behavior"],
        ["https://www.example-shop.com/au/apps/wallet/", "Live", False, None, None, "analytics request not confidently captured"],
        ["https://www.example-shop.com/au/info/privacy/", "Live", True, "info", None, "site-section missing"],
        ["https://www.example-shop.com/us/support/legacy-device/", "404", False, None, None, "dead page"],
        ["https://www.example-shop.com/de/unknown/landing/", "Review", False, None, None, "requires manual inspection"],
        ["not a valid url", "Malformed", False, None, None, "invalid URL structure"],
    ],
    columns=[
        "normalized_url",
        "url_status",
        "first_pv_captured",
        "observed_page_track",
        "observed_site_section",
        "inspection_note",
    ],
)

assert inspection_df["normalized_url"].is_unique
display(inspection_df)


,normalized_url,url_status,first_pv_captured,observed_page_track,observed_site_section,inspection_note
0,https://www.example-shop.com/us/support/phones/,Live,True,support,support,expected capture
1,https://www.example-shop.com/de/shop/smartphones/,Live,True,product,product,site-section mismatch
2,https://www.example-shop.com/sg/offers/student/,Live,True,None,offers,page-track missing
3,https://www.example-shop.com/sg/offers/student/?ref=partner,Live,True,offers,offers,query changes page behavior
4,https://www.example-shop.com/au/apps/wallet/,Live,False,None,None,analytics request not confidently captured
5,https://www.example-shop.com/au/info/privacy/,Live,True,info,None,site-section missing
6,https://www.example-shop.com/us/support/legacy-device/,404,False,None,None,dead page
7,https://www.example-shop.com/de/unknown/landing/,Review,False,None,None,requires manual inspection
8,not a valid url,Malformed,False,None,None,invalid URL structure


## 5. Measurement classification

Classification is intentionally conservative:

- `404` / `Malformed` → **error traffic**
- non-Live or no confident first page-view → **manual review**
- Live + first page-view + missing/mismatched required fields → **measurement fix**
- otherwise → **healthy measurement**

The manual-review branch is important: an uncertain capture should not automatically be labeled a tagging defect.


In [5]:
MISSING_TOKENS = {
    "", "undefined", "null", "none", "not set", "nan", "<na>",
}


def missing_mask(series: pd.Series) -> pd.Series:
    normalized = series.astype("string").str.strip().str.lower()
    return series.isna() | normalized.isin(MISSING_TOKENS)


def classify_measurement_audit(audit_df: pd.DataFrame, checks_df: pd.DataFrame) -> pd.DataFrame:
    if audit_df["normalized_url"].duplicated().any():
        raise ValueError("audit_df must contain one row per normalized_url")
    if checks_df["normalized_url"].duplicated().any():
        raise ValueError("checks_df must contain one row per normalized_url")

    df = audit_df.merge(
        checks_df,
        on="normalized_url",
        how="left",
        validate="one_to_one",
    )

    df["first_pv_captured"] = df["first_pv_captured"].fillna(False).astype(bool)

    page_track_missing = missing_mask(df["observed_page_track"])
    expected_missing = missing_mask(df["expected_site_section"])
    observed_section_missing = missing_mask(df["observed_site_section"])

    expected_norm = df["expected_site_section"].astype("string").str.strip().str.lower()
    observed_norm = df["observed_site_section"].astype("string").str.strip().str.lower()

    section_match = (
        ~expected_missing
        & ~observed_section_missing
        & expected_norm.eq(observed_norm)
    )

    df["page_track_check"] = np.where(page_track_missing, "missing", "captured")
    df["site_section_check"] = np.select(
        [expected_missing, observed_section_missing, section_match],
        ["no_expectation", "missing", "match"],
        default="mismatch",
    )

    status = df["url_status"].astype("string")
    error_traffic = status.isin(["404", "Malformed"])
    manual_review = ~status.eq("Live") | ~df["first_pv_captured"]

    section_bad = df["site_section_check"].isin(["missing", "mismatch"])
    measurement_fix = (
        status.eq("Live")
        & df["first_pv_captured"]
        & (page_track_missing | section_bad)
    )

    df["disposition"] = np.select(
        [error_traffic, manual_review, measurement_fix],
        ["error_traffic", "manual_review", "measurement_fix"],
        default="healthy",
    )

    reasons = pd.Series(pd.NA, index=df.index, dtype="string")
    reasons.loc[page_track_missing & section_bad] = "page_track missing / site_section invalid"
    reasons.loc[page_track_missing & ~section_bad] = "page_track missing"
    reasons.loc[~page_track_missing & df["site_section_check"].eq("missing")] = "site_section missing"
    reasons.loc[~page_track_missing & df["site_section_check"].eq("mismatch")] = "site_section mismatch"
    df["fix_reason"] = reasons

    return df


audit_df = classify_measurement_audit(audit_input_df, inspection_df)

display(
    audit_df[
        [
            "normalized_url",
            "instances",
            "url_status",
            "first_pv_captured",
            "expected_site_section",
            "observed_site_section",
            "page_track_check",
            "site_section_check",
            "disposition",
            "fix_reason",
        ]
    ].sort_values("instances", ascending=False)
)


,normalized_url,instances,url_status,first_pv_captured,expected_site_section,observed_site_section,page_track_check,site_section_check,disposition,fix_reason
0,https://www.example-shop.com/de/shop/smartphones/,9700,Live,True,shop,product,captured,mismatch,measurement_fix,site_section mismatch
1,https://www.example-shop.com/us/support/phones/,8400,Live,True,support,support,captured,match,healthy,<NA>
2,https://www.example-shop.com/sg/offers/student/,5100,Live,True,offers,offers,missing,match,measurement_fix,page_track missing
3,https://www.example-shop.com/au/apps/wallet/,3300,Live,False,apps,None,missing,missing,manual_review,page_track missing / site_section invalid
4,https://www.example-shop.com/au/info/privacy/,1600,Live,True,info,None,captured,missing,measurement_fix,site_section missing
5,https://www.example-shop.com/sg/offers/student/?ref=partner,900,Live,True,offers,offers,captured,match,healthy,<NA>
6,https://www.example-shop.com/us/support/legacy-device/,700,404,False,support,None,missing,missing,error_traffic,page_track missing / site_section invalid
7,https://www.example-shop.com/de/unknown/landing/,450,Review,False,unknown,None,missing,missing,manual_review,page_track missing / site_section invalid
8,not a valid url,120,Malformed,False,NaN,None,missing,no_expectation,error_traffic,page_track missing


## 6. Convert defects into remediation units

Fixing every URL independently is inefficient when many URLs share the same implementation surface.

The grouping rule uses the first three path segments:

`/{market}/{page_section}/{page_family}/`

Within each issue group, the URL with the largest traffic volume becomes the representative reproduction URL.


In [6]:
def derive_group_path(url):
    if pd.isna(url):
        return pd.NA

    try:
        parts = urlsplit(str(url))
        if not parts.hostname:
            return pd.NA

        segments = [
            segment
            for segment in re.sub(r"/{2,}", "/", parts.path or "/").strip("/").split("/")
            if segment
        ]
    except Exception:
        return pd.NA

    if not segments:
        return "/"

    return "/" + "/".join(segments[:3]) + "/"


audit_df["group_path"] = audit_df["normalized_url"].apply(derive_group_path)

fix_detail_df = audit_df.loc[
    audit_df["disposition"].eq("measurement_fix")
].copy()

issue_keys = [
    "market",
    "group_path",
    "expected_site_section",
    "fix_reason",
]

representative_df = (
    fix_detail_df
    .sort_values(["instances", "normalized_url"], ascending=[False, True])
    .drop_duplicates(issue_keys, keep="first")
    [issue_keys + ["normalized_url", "observed_page_track", "observed_site_section"]]
    .rename(columns={"normalized_url": "representative_url"})
)

remediation_df = (
    fix_detail_df
    .groupby(issue_keys, dropna=False, as_index=False)
    .agg(
        affected_instances=("instances", "sum"),
        affected_url_count=("normalized_url", "nunique"),
        source_row_count=("source_row_count", "sum"),
    )
    .merge(representative_df, on=issue_keys, how="left", validate="one_to_one")
    .sort_values("affected_instances", ascending=False)
    .reset_index(drop=True)
)

display(remediation_df)


,market,group_path,expected_site_section,fix_reason,affected_instances,affected_url_count,source_row_count,representative_url,observed_page_track,observed_site_section
0,de,/de/shop/smartphones/,shop,site_section mismatch,9700,1,2,https://www.example-shop.com/de/shop/smartphones/,product,product
1,sg,/sg/offers/student/,offers,page_track missing,5100,1,1,https://www.example-shop.com/sg/offers/student/,None,offers
2,au,/au/info/privacy/,info,site_section missing,1600,1,1,https://www.example-shop.com/au/info/privacy/,info,None


## 7. Validation and impact summary

A measurement pipeline should validate its own transformations.

The checks below verify:

- normalized URLs remain unique after aggregation,
- total traffic is conserved,
- every row receives exactly one disposition,
- remediation rows are a strict subset of confidently captured Live pages,
- grouped remediation traffic equals the underlying defect traffic.


In [7]:
input_instances = working_df["instances"].sum()
audit_instances = audit_df["instances"].sum()
fix_instances = fix_detail_df["instances"].sum()
grouped_fix_instances = remediation_df["affected_instances"].sum()

assert audit_df["normalized_url"].is_unique
assert np.isclose(input_instances, audit_instances)
assert audit_df["disposition"].notna().all()
assert set(fix_detail_df["url_status"]) == {"Live"} if len(fix_detail_df) else True
assert fix_detail_df["first_pv_captured"].all() if len(fix_detail_df) else True
assert np.isclose(fix_instances, grouped_fix_instances)

summary_df = (
    audit_df
    .groupby("disposition", as_index=False)
    .agg(
        url_count=("normalized_url", "nunique"),
        instances=("instances", "sum"),
    )
)

summary_df["traffic_share"] = summary_df["instances"] / summary_df["instances"].sum()
summary_df = summary_df.sort_values("instances", ascending=False).reset_index(drop=True)

display(summary_df)

print(f"Total instances preserved: {input_instances:,.0f}")
print(f"Measurement-fix traffic:   {fix_instances:,.0f}")
print(f"Remediation groups:        {len(remediation_df):,}")


,disposition,url_count,instances,traffic_share
0,measurement_fix,3,16400,0.541791
1,healthy,2,9300,0.307235
2,manual_review,2,3750,0.123885
3,error_traffic,2,820,0.027090


Total instances preserved: 30,270
Measurement-fix traffic:   16,400
Remediation groups:        3


## 8. Interpretation

This workflow does more than flag missing values.

It separates four analytically different situations:

- **Healthy measurement:** the page is Live, the first page-view is confidently observed, and required fields match expectations.
- **Measurement fix:** the page is Live and observable, but a required field is missing or inconsistent.
- **Error traffic:** the URL itself is invalid or dead; this should not be reported as a tagging defect.
- **Manual review:** the automated capture is not reliable enough to support a conclusion.

The remediation table then prioritizes issues using **traffic exposure**, while preserving a representative URL that engineering or QA can use for reproduction.

### What this demonstrates

- URL canonicalization and feature extraction
- vectorized data-quality rules
- explicit expected-vs-observed validation
- conservative handling of uncertain evidence
- weighted impact prioritization
- issue deduplication into remediation units
- reproducibility through data-conservation assertions
- separation of data acquisition from analytical decision logic

### What this notebook does *not* claim

The browser/network capture layer is abstracted, and the synthetic examples do not represent any real company implementation. The notebook demonstrates the **analytical framework**, not a proprietary production tagging specification.
